## Aplicação da extrategia de investimento do Greenblatt



## Critérios
Fazer 2 colunas ordenas:

    - Coluna 1: Colocará 0 para a empresa que tiver o maior EV/EBIT;

    - Coluna 2: Colocará 0 no que tem o menor ROE = LUCRO_LIQUIDO/ PL;

    - Coluna 3: somara os valores das colunas anteriores e ordenará da maior pontuação para a menor.

No metodo diz para avaliar os primeiros ativos segundo o metodo, mas para executar este trabalho será escolhido os 10 primerios ativos

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf
import investpy as inv
from datetime import timedelta


In [2]:
bolsa_br = inv.get_stocks(country= 'Brazil')
ticket = bolsa_br['symbol']
ticket

0       ABCB4
1       AGRO3
2       RAIL3
3       ALPA3
4       ALPA4
        ...  
744    IBFF11
745    CVBI11
746    HRDF11
747    RSPD11
748    TCPF11
Name: symbol, Length: 749, dtype: object

In [3]:
acoes = [acao+'.SA' for acao in ticket if len(acao) < 6]
print(len(acoes))

390


In [4]:
# De acordo com o metodo de GreenBlatt ele elimina empresa do setor financeiros por não
# Possuirem certos indicadores
acoes_financeiras_filtradas =[]
for acao in acoes:
    empresa = yf.Ticker(acao)
    try:
        informacoes = empresa.info
        if informacoes:
            setor = informacoes.get("sector")
            if setor != "Financial Services":
                acoes_financeiras_filtradas.append(acao)
    except:
        continue
print(len(acoes_financeiras_filtradas))

404 Client Error: Not Found for url: https://query2.finance.yahoo.com/v10/finance/quoteSummary/ALSO3.SA?modules=financialData%2CquoteType%2CdefaultKeyStatistics%2CassetProfile%2CsummaryDetail&corsDomain=finance.yahoo.com&formatted=false&symbol=ALSO3.SA&crumb=VcWqfNbqnIm
404 Client Error: Not Found for url: https://query2.finance.yahoo.com/v10/finance/quoteSummary/ARZZ3.SA?modules=financialData%2CquoteType%2CdefaultKeyStatistics%2CassetProfile%2CsummaryDetail&corsDomain=finance.yahoo.com&formatted=false&symbol=ARZZ3.SA&crumb=VcWqfNbqnIm
404 Client Error: Not Found for url: https://query2.finance.yahoo.com/v10/finance/quoteSummary/APER3.SA?modules=financialData%2CquoteType%2CdefaultKeyStatistics%2CassetProfile%2CsummaryDetail&corsDomain=finance.yahoo.com&formatted=false&symbol=APER3.SA&crumb=VcWqfNbqnIm
404 Client Error: Not Found for url: https://query2.finance.yahoo.com/v10/finance/quoteSummary/CIEL3.SA?modules=financialData%2CquoteType%2CdefaultKeyStatistics%2CassetProfile%2CsummaryDe

292


In [5]:
df_acoes = pd.DataFrame(acoes_financeiras_filtradas, columns= ['Ticker'])
df_acoes

,Ticker
0,AGRO3.SA
1,RAIL3.SA
2,ALPA3.SA
3,ALPA4.SA
4,AMAR3.SA
...,...
287,RSUL4.SA
288,VAMO3.SA
289,MOAR3.SA
290,VIVA3.SA


In [6]:
data_final = '2024-12-30'
data_final = pd.to_datetime(data_final)
PERIODO =3
data_inicio = data_final - timedelta(days=365*PERIODO)
print(data_inicio)

2021-12-31 00:00:00


In [7]:
# Pegar o preço medio das ações para fazer o indicador p/l
contador = 0
while contador < len(df_acoes):
    empresa = yf.Ticker(df_acoes.loc[contador,'Ticker'])
    precos = empresa.history(start = data_inicio, end= data_final, period= '1y')
    preco_medio = precos['Close'].mean()
    df_acoes.loc[contador, "Preco_Medio"] = preco_medio
    contador += 1
df_acoes

$BBRK3.SA: possibly delisted; no timezone found
$BRML3.SA: possibly delisted; no timezone found
$BTOW3.SA: possibly delisted; no timezone found
$CAMB4.SA: possibly delisted; no timezone found
$CARD3.SA: possibly delisted; no timezone found
$CCPR3.SA: possibly delisted; no timezone found
$CESP6.SA: possibly delisted; no timezone found
$CRDE3.SA: possibly delisted; no timezone found
$DTEX3.SA: possibly delisted; no timezone found
$TIET3.SA: possibly delisted; no timezone found
$TIET4.SA: possibly delisted; no timezone found
$HGTX3.SA: possibly delisted; no timezone found
$IGTA3.SA: possibly delisted; no timezone found
$LAME3.SA: possibly delisted; no timezone found
$LAME4.SA: possibly delisted; no timezone found
$LLIS3.SA: possibly delisted; no timezone found
$LREN3.SA: possibly delisted; no price data found  (1d 2021-12-31 00:00:00 -> 2024-12-30 00:00:00)
$MMXM3.SA: possibly delisted; no timezone found
$TESA3.SA: possibly delisted; no timezone found
$CESP3.SA: possibly delisted; no time

,Ticker,Preco_Medio
0,AGRO3.SA,22.516663
1,RAIL3.SA,19.964849
2,ALPA3.SA,12.153967
3,ALPA4.SA,13.008049
4,AMAR3.SA,5.850405
...,...,...
287,RSUL4.SA,65.768737
288,VAMO3.SA,9.858616
289,MOAR3.SA,373.430621
290,VIVA3.SA,24.923438


In [8]:
# Tirar as ações que não foram encontradas preços
emp_maior_q_zero = df_acoes[df_acoes['Preco_Medio']>0].reset_index(drop=True)
len(emp_maior_q_zero)

254

In [9]:
contador = 0
while contador < len(emp_maior_q_zero):
    empresa = yf.Ticker(emp_maior_q_zero.loc[contador,"Ticker"])
    dados_financeiros = empresa.financials
    lucro_medio = dados_financeiros.loc['Net Income'][:3].mean() if 'Net Income' in dados_financeiros.index else 0
    emp_maior_q_zero.loc[contador, 'Lucro_medio'] = lucro_medio
    contador +=1
emp_maior_q_zero

,Ticker,Preco_Medio,Lucro_medio
0,AGRO3.SA,22.516663,3.385010e+08
1,RAIL3.SA,19.964849,9.178633e+07
2,ALPA3.SA,12.153967,-5.457263e+08
3,ALPA4.SA,13.008049,-5.457263e+08
4,AMAR3.SA,5.850405,-3.718700e+08
...,...,...,...
249,RSUL4.SA,65.768737,6.994167e+07
250,VAMO3.SA,9.858616,5.526543e+08
251,MOAR3.SA,373.430621,3.941300e+08
252,VIVA3.SA,24.923438,3.428631e+08


In [10]:
emp_lucro_maior_0 = emp_maior_q_zero[emp_maior_q_zero['Lucro_medio']>0].reset_index(drop=True)
emp_lucro_maior_0

,Ticker,Preco_Medio,Lucro_medio
0,AGRO3.SA,22.516663,3.385010e+08
1,RAIL3.SA,19.964849,9.178633e+07
2,ABEV3.SA,12.418805,1.387693e+10
3,BEEF3.SA,9.326857,5.574887e+08
4,BRKM3.SA,26.247896,3.023333e+09
...,...,...,...
190,RSUL4.SA,65.768737,6.994167e+07
191,VAMO3.SA,9.858616,5.526543e+08
192,MOAR3.SA,373.430621,3.941300e+08
193,VIVA3.SA,24.923438,3.428631e+08


In [11]:
# Calculo do pl = qtd_acoes* valor patrimonial
contador = 0
while contador < len(emp_lucro_maior_0):
    empresa = yf.Ticker(emp_lucro_maior_0.loc[contador,"Ticker"])
    informacoes = empresa.info
    acoes = informacoes.get('sharesOutstanding')
    valor_patrimonial = informacoes.get('bookValue')
    emp_lucro_maior_0.loc[contador, 'Valor_patrimonial'] = valor_patrimonial
    emp_lucro_maior_0.loc[contador, 'Qtd_acoes'] = acoes
    contador +=1
emp_lucro_maior_0

,Ticker,Preco_Medio,Lucro_medio,Valor_patrimonial,Qtd_acoes
0,AGRO3.SA,22.516663,3.385010e+08,22.112,9.961550e+07
1,RAIL3.SA,19.964849,9.178633e+07,7.960,1.850710e+09
2,ABEV3.SA,12.418805,1.387693e+10,6.272,1.568880e+10
3,BEEF3.SA,9.326857,5.574887e+08,1.265,5.883260e+08
4,BRKM3.SA,26.247896,3.023333e+09,-6.000,4.516690e+08
...,...,...,...,...,...
190,RSUL4.SA,65.768737,6.994167e+07,36.352,2.495220e+06
191,VAMO3.SA,9.858616,5.526543e+08,4.767,1.078060e+09
192,MOAR3.SA,373.430621,3.941300e+08,110.438,1.225120e+07
193,VIVA3.SA,24.923438,3.428631e+08,9.712,2.350630e+08


In [12]:
# CALCULO DO EV (ENTERPRISE VALUE) = PRECO DE FIRMA+ DIVIDA LIQUIDA
contador = 0
while contador < len(emp_lucro_maior_0):
    empresa = yf.Ticker(emp_lucro_maior_0.loc[contador,"Ticker"])
    balanco = empresa.balance_sheet
    total_divida = balanco.loc["Total Debt"].mean() if "Total Debt" in balanco.index else 0
    caixa = balanco.loc['Cash And Cash Equivalents'].mean() if 'Cash And Cash Equivalents' in balanco.index else 0
    emp_lucro_maior_0.loc[contador, 'Divida_liquida'] = total_divida-caixa
    contador+=1
emp_lucro_maior_0

,Ticker,Preco_Medio,Lucro_medio,Valor_patrimonial,Qtd_acoes,Divida_liquida
0,AGRO3.SA,22.516663,3.385010e+08,22.112,9.961550e+07,3.577368e+08
1,RAIL3.SA,19.964849,9.178633e+07,7.960,1.850710e+09,1.799695e+10
2,ABEV3.SA,12.418805,1.387693e+10,6.272,1.568880e+10,-1.235848e+10
3,BEEF3.SA,9.326857,5.574887e+08,1.265,5.883260e+08,6.752386e+09
4,BRKM3.SA,26.247896,3.023333e+09,-6.000,4.516690e+08,4.480100e+10
...,...,...,...,...,...,...
190,RSUL4.SA,65.768737,6.994167e+07,36.352,2.495220e+06,9.183750e+06
191,VAMO3.SA,9.858616,5.526543e+08,4.767,1.078060e+09,6.949784e+09
192,MOAR3.SA,373.430621,3.941300e+08,110.438,1.225120e+07,4.067850e+08
193,VIVA3.SA,24.923438,3.428631e+08,9.712,2.350630e+08,4.164962e+08


In [13]:
contador=0
while contador < len(emp_lucro_maior_0):
    empresa = yf.Ticker(emp_lucro_maior_0.loc[contador,"Ticker"])
    dados_financeiros = empresa.financials
    ebit = dados_financeiros.loc['EBIT'].mean() if 'EBIT' in dados_financeiros.index else 0
    emp_lucro_maior_0.loc[contador, 'EBIT'] = ebit
    contador+=1
emp_lucro_maior_0

,Ticker,Preco_Medio,Lucro_medio,Valor_patrimonial,Qtd_acoes,Divida_liquida,EBIT
0,AGRO3.SA,22.516663,3.385010e+08,22.112,9.961550e+07,3.577368e+08,4.060388e+08
1,RAIL3.SA,19.964849,9.178633e+07,7.960,1.850710e+09,1.799695e+10,3.472408e+09
2,ABEV3.SA,12.418805,1.387693e+10,6.272,1.568880e+10,-1.235848e+10,1.654590e+10
3,BEEF3.SA,9.326857,5.574887e+08,1.265,5.883260e+08,6.752386e+09,1.748411e+09
4,BRKM3.SA,26.247896,3.023333e+09,-6.000,4.516690e+08,4.480100e+10,4.479621e+09
...,...,...,...,...,...,...,...
190,RSUL4.SA,65.768737,6.994167e+07,36.352,2.495220e+06,9.183750e+06,9.050975e+07
191,VAMO3.SA,9.858616,5.526543e+08,4.767,1.078060e+09,6.949784e+09,1.282175e+09
192,MOAR3.SA,373.430621,3.941300e+08,110.438,1.225120e+07,4.067850e+08,5.799762e+08
193,VIVA3.SA,24.923438,3.428631e+08,9.712,2.350630e+08,4.164962e+08,3.627269e+08


In [14]:
df_acoes_vpa_maior_0 = emp_lucro_maior_0[emp_lucro_maior_0['Valor_patrimonial']>0]
df_ebit_maior_0 = df_acoes_vpa_maior_0[df_acoes_vpa_maior_0['EBIT']>0]
len(df_ebit_maior_0)

177

In [15]:
len(df_acoes_vpa_maior_0)

182

In [16]:
# Calcular ROE = lucro/ pl, e o PL = qtd_acoes*vpa
#EV/EBIT = MARKETCAP+DIVIDA_LIQUIDA/EBIT
df_ebit_maior_0['ROE'] = (df_ebit_maior_0['Lucro_medio']/(df_ebit_maior_0['Qtd_acoes']*df_ebit_maior_0['Valor_patrimonial']))*100
df_ebit_maior_0["EV"]= (df_ebit_maior_0['Preco_Medio']* df_ebit_maior_0['Qtd_acoes']) +df_ebit_maior_0['Divida_liquida']
df_ebit_maior_0

C:\Users\edens\AppData\Local\Temp\ipykernel_7828\3881198216.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_ebit_maior_0['ROE'] = (df_ebit_maior_0['Lucro_medio']/(df_ebit_maior_0['Qtd_acoes']*df_ebit_maior_0['Valor_patrimonial']))*100
C:\Users\edens\AppData\Local\Temp\ipykernel_7828\3881198216.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_ebit_maior_0["EV"]= (df_ebit_maior_0['Preco_Medio']* df_ebit_maior_0['Qtd_acoes']) +df_ebit_maior_0['Divida_liquida']


,Ticker,Preco_Medio,Lucro_medio,Valor_patrimonial,Qtd_acoes,Divida_liquida,EBIT,ROE,EV
0,AGRO3.SA,22.516663,3.385010e+08,22.112,9.961550e+07,3.577368e+08,4.060388e+08,15.367563,2.600745e+09
1,RAIL3.SA,19.964849,9.178633e+07,7.960,1.850710e+09,1.799695e+10,3.472408e+09,0.623055,5.494610e+10
2,ABEV3.SA,12.418805,1.387693e+10,6.272,1.568880e+10,-1.235848e+10,1.654590e+10,14.102552,1.824777e+11
3,BEEF3.SA,9.326857,5.574887e+08,1.265,5.883260e+08,6.752386e+09,1.748411e+09,74.907873,1.223962e+10
6,CCRO3.SA,12.176662,2.362208e+09,6.770,2.010120e+09,2.572149e+10,6.105873e+09,17.358310,5.019805e+10
...,...,...,...,...,...,...,...,...,...
190,RSUL4.SA,65.768737,6.994167e+07,36.352,2.495220e+06,9.183750e+06,9.050975e+07,77.107891,1.732912e+08
191,VAMO3.SA,9.858616,5.526543e+08,4.767,1.078060e+09,6.949784e+09,1.282175e+09,10.753887,1.757796e+10
192,MOAR3.SA,373.430621,3.941300e+08,110.438,1.225120e+07,4.067850e+08,5.799762e+08,29.130124,4.981758e+09
193,VIVA3.SA,24.923438,3.428631e+08,9.712,2.350630e+08,4.164962e+08,3.627269e+08,15.018542,6.275075e+09


In [17]:
df_ebit_maior_0["EV/EBIT"] = df_ebit_maior_0['EV']/ df_ebit_maior_0['EBIT']

C:\Users\edens\AppData\Local\Temp\ipykernel_7828\704606420.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_ebit_maior_0["EV/EBIT"] = df_ebit_maior_0['EV']/ df_ebit_maior_0['EBIT']


In [18]:
df_filtro = df_ebit_maior_0[["Ticker",'EV/EBIT', 'ROE']].reset_index(drop=True)
df_filtro

,Ticker,EV/EBIT,ROE
0,AGRO3.SA,6.405166,15.367563
1,RAIL3.SA,15.823632,0.623055
2,ABEV3.SA,11.028573,14.102552
3,BEEF3.SA,7.000425,74.907873
4,CCRO3.SA,8.221272,17.358310
...,...,...,...
172,RSUL4.SA,1.914614,77.107891
173,VAMO3.SA,13.709486,10.753887
174,MOAR3.SA,8.589590,29.130124
175,VIVA3.SA,17.299720,15.018542


In [19]:
# Roe acima de 20 desconsiderar devido ao fator de não realidade 
df_filtro_ev_ebit_negativo = df_filtro[df_filtro["EV/EBIT"]>0]

In [20]:
len(df_filtro_ev_ebit_negativo)

175

In [21]:
# Quanto maior ROE maior estão no ranking
ord_elimar_roe_maior = df_filtro_ev_ebit_negativo[df_filtro_ev_ebit_negativo['ROE']<20]
ord_ROE=ord_elimar_roe_maior.sort_values('ROE', ascending= False)
ord_ROE['indice_ROE']= range(len(ord_ROE))

In [22]:
ord_ROE

,Ticker,EV/EBIT,ROE,indice_ROE
5,CEDO4.SA,5.006109,19.968129,0
106,SAPR4.SA,3.719821,19.392989,1
13,CPLE6.SA,6.516171,19.241510,2
157,TAEE4.SA,4.369456,18.744023,3
56,MULT3.SA,11.216071,18.482861,4
...,...,...,...,...
54,ENEV3.SA,24.374701,3.243400,85
103,GUAR3.SA,14.028755,3.044187,86
24,YDUQ3.SA,12.937208,2.786834,87
1,RAIL3.SA,15.823632,0.623055,88


In [23]:
ord_EV_EBIT = ord_ROE.sort_values('EV/EBIT', ascending=True)
ord_EV_EBIT['indice_EV_EBIT']= range(len(ord_EV_EBIT))

In [24]:
ord_EV_EBIT

,Ticker,EV/EBIT,ROE,indice_ROE,indice_EV_EBIT
107,CRIV4.SA,1.039492,9.823598,55,0
116,CRIV3.SA,1.161315,7.391214,67,1
86,USIM5.SA,1.424741,9.305690,58,2
85,USIM3.SA,1.695549,6.993722,70,3
119,DOHL4.SA,2.279397,14.178978,26,4
...,...,...,...,...,...
54,ENEV3.SA,24.374701,3.243400,85,85
96,RADL3.SA,26.144955,16.844764,11,86
162,AZEV3.SA,32.000458,12.455658,41,87
50,LUPA3.SA,32.947466,18.354043,5,88


In [25]:
ord_EV_EBIT["Indice_Greenblatt"]= ord_EV_EBIT["indice_EV_EBIT"]+ ord_ROE['indice_ROE']
ord_acoes = ord_EV_EBIT.sort_values('Indice_Greenblatt', ascending=True).reset_index(drop=True)
ord_acoes

,Ticker,EV/EBIT,ROE,indice_ROE,indice_EV_EBIT,Indice_Greenblatt
0,SAPR4.SA,3.719821,19.392989,1,10,11
1,TAEE4.SA,4.369456,18.744023,3,14,17
2,EMAE4.SA,3.729954,17.734167,8,11,19
3,CEDO4.SA,5.006109,19.968129,0,19,19
4,PTNT4.SA,3.386479,15.172505,18,9,27
...,...,...,...,...,...,...
85,GUAR3.SA,14.028755,3.044187,86,74,160
86,AZEV4.SA,55.273260,6.437880,74,89,163
87,RAIL3.SA,15.823632,0.623055,88,79,167
88,TPIS3.SA,16.253781,0.385628,89,80,169


In [26]:
ord_acoes['empresa']=ord_acoes['Ticker'].str[:4]
ord_acoes_carteira = ord_acoes.sort_values('Indice_Greenblatt', ascending=True)
ord_acoes= ord_acoes_carteira.drop_duplicates('empresa', keep='first').head(10)
ord_acoes

,Ticker,EV/EBIT,ROE,indice_ROE,indice_EV_EBIT,Indice_Greenblatt,empresa
0,SAPR4.SA,3.719821,19.392989,1,10,11,SAPR
1,TAEE4.SA,4.369456,18.744023,3,14,17,TAEE
2,EMAE4.SA,3.729954,17.734167,8,11,19,EMAE
3,CEDO4.SA,5.006109,19.968129,0,19,19,CEDO
4,PTNT4.SA,3.386479,15.172505,18,9,27,PTNT
5,AHEB3.SA,4.609837,16.075536,13,15,28,AHEB
6,GEPA4.SA,2.821368,14.379989,24,5,29,GEPA
7,DOHL4.SA,2.279397,14.178978,26,4,30,DOHL
8,POSI3.SA,3.866032,15.140775,19,12,31,POSI
9,PETR4.SA,5.349682,17.539498,9,24,33,PETR


In [27]:
CARTEIRA_GREENBLATT = ord_acoes[['Ticker']]
CARTEIRA_GREENBLATT

,Ticker
0,SAPR4.SA
1,TAEE4.SA
2,EMAE4.SA
3,CEDO4.SA
4,PTNT4.SA
5,AHEB3.SA
6,GEPA4.SA
7,DOHL4.SA
8,POSI3.SA
9,PETR4.SA


In [28]:
def investimento(ano):
    SALARIOS = {2000:151,2001:180,2002:200, 2003:240, 2004:260, 2005:300,2006:350,2007:380,2008:415,2009:465,2010:510,2011:540,2012:622,2013:678,
            2014:724,2015:788,2016:880,2017:937,2018:954,2019:998,2020:1039,2021:1100,2022:1212,2023:1302,2024:1412}
    TAXA_INVEST = 0.075
    investimento = SALARIOS[ano]*TAXA_INVEST
    return round(investimento,2)


In [29]:
# Tempo do backtest
ano_inicio = '2000-01-01'
#ano_inicio = pd.to_datetime(ano_inicio)
data_final= '2024-12-31'
#data_final= pd.to_datetime(ano_inicio)

In [30]:
precos_ativos = {}
preco_acoes={}
for acoes in CARTEIRA_GREENBLATT['Ticker']:
    empresa= yf.Ticker(acoes)
    precos = empresa.history(start=ano_inicio,end=data_final, period='1mo')
    preco_fechamento = precos['Close']
    preco_acoes[acoes] = preco_fechamento.resample('M').last()
bolsa_greenblatt = pd.DataFrame(preco_acoes).reset_index()


C:\Users\edens\AppData\Local\Temp\ipykernel_7828\2525569431.py:7: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  preco_acoes[acoes] = preco_fechamento.resample('M').last()
C:\Users\edens\AppData\Local\Temp\ipykernel_7828\2525569431.py:7: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  preco_acoes[acoes] = preco_fechamento.resample('M').last()
C:\Users\edens\AppData\Local\Temp\ipykernel_7828\2525569431.py:7: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  preco_acoes[acoes] = preco_fechamento.resample('M').last()
C:\Users\edens\AppData\Local\Temp\ipykernel_7828\2525569431.py:7: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  preco_acoes[acoes] = preco_fechamento.resample('M').last()
C:\Users\edens\AppData\Local\Temp\ipykernel_7828\2525569431.py:7: FutureWarning: 'M' is deprecat

In [31]:
bolsa_greenblatt

,Date,SAPR4.SA,TAEE4.SA,EMAE4.SA,CEDO4.SA,PTNT4.SA,AHEB3.SA,GEPA4.SA,DOHL4.SA,POSI3.SA,PETR4.SA
0,2000-01-31 00:00:00-02:00,NaN,NaN,2.400970,NaN,NaN,2.219244,2.615628,0.868057,NaN,1.117574
1,2000-02-29 00:00:00-03:00,NaN,NaN,3.001212,NaN,NaN,2.226641,2.395364,0.868057,NaN,1.265240
2,2000-03-31 00:00:00-03:00,NaN,NaN,2.995210,NaN,NaN,2.234039,1.927304,0.868057,NaN,1.292627
3,2000-04-30 00:00:00-03:00,NaN,NaN,2.460994,NaN,NaN,2.234039,2.285232,1.041468,NaN,1.177824
4,2000-05-31 00:00:00-03:00,NaN,NaN,2.551030,NaN,NaN,2.234039,1.764860,1.041468,NaN,1.147589
...,...,...,...,...,...,...,...,...,...,...,...
295,2024-08-31 00:00:00-03:00,5.680338,11.655107,40.533104,24.500000,6.965643,35.000000,24.869888,4.370000,6.53,36.488823
296,2024-09-30 00:00:00-03:00,5.787148,11.262679,38.880001,23.000000,6.332402,37.500000,26.328558,4.190000,5.68,33.374714
297,2024-10-31 00:00:00-03:00,5.379329,11.429461,39.779999,24.459999,6.280000,38.000000,29.190403,4.070000,6.41,33.282032
298,2024-11-30 00:00:00-03:00,5.884248,11.390000,39.990002,22.000000,6.180000,38.000000,29.634438,4.120000,5.22,36.053219


In [32]:

contador = 0
saldo = 0
carteira_contabil_greenblatt = 0
valor_investido_greenblatt = 0
compras_greenblatt= {acao:0 for acao in CARTEIRA_GREENBLATT['Ticker']}
while contador < len(bolsa_greenblatt):
    data=bolsa_greenblatt.loc[contador, 'Date'].year
    saldo += investimento(data)
    valor_investido_greenblatt+= investimento(data)
    while True:
        compra_realizada=False
        for acao in CARTEIRA_GREENBLATT['Ticker']:
            preco = bolsa_greenblatt.loc[contador, acao]
            if saldo > preco and preco >0:
                compras_greenblatt[acao] +=1
                saldo -= preco
                carteira_contabil_greenblatt += preco
                compra_realizada = True
        if not compra_realizada:
            break
    contador +=1
print(compras_greenblatt)
print(carteira_contabil_greenblatt)
print(valor_investido_greenblatt)


{'SAPR4.SA': 453, 'TAEE4.SA': 112, 'EMAE4.SA': 391, 'CEDO4.SA': 216, 'PTNT4.SA': 330, 'AHEB3.SA': 222, 'GEPA4.SA': 231, 'DOHL4.SA': 411, 'POSI3.SA': 133, 'PETR4.SA': 172}
14972.274313300848
14973.120000000006


In [33]:
file = pd.ExcelWriter('Arquivo_Greenblatt.xlsx', engine='openpyxl')
bolsa_greenblatt_salv= bolsa_greenblatt.fillna(0)
bolsa_greenblatt_salv['Date'] = bolsa_greenblatt['Date'].dt.strftime('%Y-%m-%d')
bolsa_greenblatt_salv.to_excel(file, sheet_name='Bolsa Greenblatt', index=False)
dicionario_inf = {
    'Nome':['carteira_contabil_greenblatt','valor_investido_greenblatt'],
    'Valor':[carteira_contabil_greenblatt,valor_investido_greenblatt]}
df_valor = pd.DataFrame(dicionario_inf)
df_valor.to_excel(file, sheet_name='valor_investido_vs_investimento', index=False)
compra_feita = {
    'Acao' : [acao for acao in compras_greenblatt.keys()],
    'Qtd_comprada': [valor for valor in compras_greenblatt.values()]}
compra_df = pd.DataFrame(compra_feita)
compra_df.to_excel(file, sheet_name='Compras', index=False)
file.close()